In [1]:

import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Generating synthetic highly imbalanced dataset (284,807 samples, 0.17% fraud)...")
# Project 2 ki strict requirement ke hisaab se data generate karna
X, y = make_classification(
    n_samples=284807,
    n_features=30,
    n_informative=20,
    n_redundant=5,
    weights=[0.9983, 0.0017], # 99.83% legitimate, 0.17% fraud
    random_state=42
)

# 80/20 Train/Test Split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# ==========================================
# 1. Logistic Regression Pipeline 
# ==========================================
lr_steps = [
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000))
]
lr_pipeline = Pipeline(steps=lr_steps)

# Time bachane ke liye grid search ke parameters thode kam kiye hain
lr_params = {
    'smote__k_neighbors': [5],
    'classifier__C': [0.1, 1.0]
}

print("Training Logistic Regression...")
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=3, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train, y_train)

# ==========================================
# 2. Random Forest Pipeline
# ==========================================
rf_steps = [
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))
]
rf_pipeline = Pipeline(steps=rf_steps)

rf_params = {
    'classifier__max_depth': [10, None]
}

print("Training Random Forest...")
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)

# ==========================================
# Final Evaluation 
# ==========================================
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f"\n--- {model_name} Evaluation ---")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("-" * 40)

evaluate_model(lr_grid.best_estimator_, X_test, y_test, "Logistic Regression")
evaluate_model(rf_grid.best_estimator_, X_test, y_test, "Random Forest")

Generating synthetic highly imbalanced dataset (284,807 samples, 0.17% fraud)...
Training Logistic Regression...
Training Random Forest...

--- Logistic Regression Evaluation ---
Precision: 0.0089
Recall: 0.5173
ROC-AUC: 0.6139

Confusion Matrix:
 [[34880 21707]
 [  181   194]]
----------------------------------------

--- Random Forest Evaluation ---
Precision: 0.0800
Recall: 0.0107
ROC-AUC: 0.6066

Confusion Matrix:
 [[56541    46]
 [  371     4]]
----------------------------------------


In [2]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Generating synthetic highly imbalanced dataset (284,807 samples, 0.17% fraud)...")
# Reduced features slightly to save memory while keeping exact row count
X, y = make_classification(
    n_samples=284807,
    n_features=20, 
    n_informative=15,
    n_redundant=2,
    weights=[0.9983, 0.0017],
    random_state=42
)

# 80/20 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# ==========================================
# 1. Logistic Regression Pipeline (Fast)
# ==========================================
lr_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('classifier', LogisticRegression(C=0.1, max_iter=500, n_jobs=-1))
])

print("Training Logistic Regression...")
lr_pipeline.fit(X_train, y_train)

# ==========================================
# 2. Random Forest Pipeline (Optimized for Speed)
# ==========================================
# n_estimators=15 and max_depth=10 ensures it trains in seconds, not minutes
rf_pipeline = Pipeline(steps=[
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('classifier', RandomForestClassifier(n_estimators=15, max_depth=10, random_state=42, n_jobs=-1))
])

print("Training Random Forest...")
rf_pipeline.fit(X_train, y_train)

# ==========================================
# Final Evaluation 
# ==========================================
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f"\n--- {model_name} Evaluation ---")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("-" * 40)

evaluate_model(lr_pipeline, X_test, y_test, "Logistic Regression")
evaluate_model(rf_pipeline, X_test, y_test, "Random Forest")

Generating synthetic highly imbalanced dataset (284,807 samples, 0.17% fraud)...
Training Logistic Regression...
Training Random Forest...

--- Logistic Regression Evaluation ---
Precision: 0.0087
Recall: 0.5561
ROC-AUC: 0.6169

Confusion Matrix:
 [[32902 23686]
 [  166   208]]
----------------------------------------

--- Random Forest Evaluation ---
Precision: 0.0086
Recall: 0.4332
ROC-AUC: 0.5823

Confusion Matrix:
 [[37878 18710]
 [  212   162]]
----------------------------------------
